# Tidy data {#sec-data-tidy}

## Introducción

En este capítulo aprenderás una forma coherente de organizar tus datos en Python usando el principio conocido como *tidy data* (datos ordenados). El formato tidy data no es adecuado para todo, pero para muchos análisis y muchos datos tabulares será lo que necesitas. Llevar tus datos a este formato requiere algo de trabajo al principio, pero ese trabajo compensa a largo plazo. Una vez que tengas tidy data, pasarás mucho menos tiempo transformando datos de una representación a otra, lo que te permitirá dedicar más tiempo a las preguntas sobre los datos que te interesan.

En este capítulo, primero aprenderás la definición de tidy data y la verás aplicada a un dataset sencillo de ejemplo. Después nos adentraremos en la herramienta principal que usarás para ordenar datos: el *melting* (fundido). El melting te permite cambiar la forma de tus datos sin cambiar ninguno de los valores. Terminaremos con una discusión sobre datos útilmente desordenados y cómo puedes crearlos si lo necesitas.

Si este capítulo te gusta especialmente y quieres aprender más sobre la teoría subyacente, puedes consultar el artículo [Tidy Data](https://www.jstatsoft.org/article/view/v059i10) publicado en el Journal of Statistical Software.


In [ ]:
# remove cell
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline

# Plot settings
plt.style.use("https://github.com/aeturrell/python4DS/raw/main/plot_style.txt")
matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

### Requisitos previos

Este capítulo usará el paquete de análisis de datos **pandas**.

## Tidy data

Hay tres características interrelacionadas que hacen que un dataset sea tidy:

1.  Cada variable es una columna; cada columna es una variable.
2.  Cada observación es una fila; cada fila es una observación.
3.  Cada valor es una celda; cada celda es un único valor.

La siguiente figura lo muestra:

![](https://d33wubrfki0l68.cloudfront.net/6f1ddb544fc5c69a2478e444ab8112fb0eea23f8/91adc/images/tidy-1.png)

¿Por qué asegurarte de que tus datos sean tidy? Hay dos ventajas principales:

1.  Hay una ventaja general en elegir una forma coherente de almacenar los datos.
    Si tienes una estructura de datos coherente, es más fácil aprender las herramientas que trabajan con ella porque tienen una uniformidad subyacente. Algunas herramientas, por ejemplo el paquete de visualización de datos **seaborn**, están diseñadas pensando en tidy data.

2.  Hay una ventaja específica en colocar las variables en columnas, porque te permite aprovechar las operaciones vectorizadas de **pandas** (operaciones que son más eficientes).


Los tidy data no serán apropiados *siempre* ni en todos los casos, pero son una opción por defecto realmente muy buena para datos tabulares. Una vez que los uses por defecto, es más fácil pensar en cómo realizar las operaciones posteriores.

Dicho esto, los tidy data son geniales, pero una de las ventajas de **pandas** frente a otras librerías de análisis de datos es que no está *demasiado* atada a los tidy data y también puede manejar sin problemas tareas incómodas de manipulación de datos que no son tidy.

Hay dos problemas comunes en los datos que se importan que hacen que no sean tidy:

1. Una variable puede estar repartida en varias columnas.
2. Una observación puede estar dispersa en varias filas.

Para lo primero, necesitamos hacer "melt" de los datos anchos, con varias columnas, para convertirlos en datos largos.

Para lo segundo, necesitamos hacer unstack o pivot de las varias filas a columnas (es decir, pasar de largo a ancho).

Veremos ambos casos a continuación.

## Herramientas para ordenar datos con **pandas**

### Melt

`melt()` puede ayudarte a pasar de datos "más anchos" a datos "más largos", y es una función *realmente* útil de recordar.

![](https://pandas.pydata.org/docs/_images/reshaping_melt.png)

Aquí tienes un ejemplo de su uso:

In [ ]:
import pandas as pd

df = pd.DataFrame(
    {
        "first": ["John", "Mary"],
        "last": ["Doe", "Bo"],
        "job": ["Nurse", "Economist"],
        "height": [5.5, 6.0],
        "weight": [130, 150],
    }
)
print("\n Unmelted: ")
print(df)
print("\n Melted: ")
df.melt(id_vars=["first", "last"], var_name="quantity", value_vars=["height", "weight"])

::: {.callout-tip title="Ejercicio"}
Realiza un `melt()` que use `job` como id en lugar de `first` y `last`.
:::

¿Qué relación tiene esto con tidy data? A veces tendrás una variable repartida en varias columnas que querrás ordenar. Veamos este ejemplo que usa casos de [tuberculosis de la Organización Mundial de la Salud](https://www.who.int/teams/global-tuberculosis-programme/data).

Primero abramos los datos y veamos el inicio del archivo.

In [ ]:
df_tb = pd.read_parquet(
    "https://github.com/aeturrell/python4DS/raw/refs/heads/main/data/who_tb_cases.parquet"
)
df_tb.head()

Puedes ver que tenemos dos columnas para una sola variable, el año. Ahora hagamos melt.

In [ ]:
df_tb.melt(
    id_vars=["country"],
    var_name="year",
    value_vars=["1999", "2000"],
    value_name="cases",
)

Ahora tenemos una observación por fila y una variable por columna: ¡tidy!

### Una forma más sencilla de pasar de ancho a largo

Si no quieres complicarte con `melt()`, también existe `wide_to_long()`, que es muy útil para casos típicos de limpieza de datos en los que tienes datos como estos:

In [ ]:
import numpy as np

df = pd.DataFrame(
    {
        "A1970": {0: "a", 1: "b", 2: "c"},
        "A1980": {0: "d", 1: "e", 2: "f"},
        "B1970": {0: 2.5, 1: 1.2, 2: 0.7},
        "B1980": {0: 3.2, 1: 1.3, 2: 0.1},
        "X": dict(zip(range(3), np.random.randn(3))),
        "id": dict(zip(range(3), range(3))),
    }
)
df

es decir, datos en los que hay distintas variables y periodos de tiempo a lo largo de las columnas. `wide_to_long()` nos permitirá indicar cuáles son los prefijos (*stubnames*) ('A', 'B'), el nombre de la variable que siempre aparece a lo largo de las columnas (aquí, un año), los valores (aquí X) y una columna de identificador.

In [ ]:
pd.wide_to_long(df, stubnames=["A", "B"], i="id", j="year")

### Stack y unstack

Stack, `stack()`, es un atajo para tomar un único tipo de variable de datos anchos desde las columnas y convertirla en un dataset en formato largo, pero con un índice adicional.

![](https://pandas.pydata.org/docs/_images/reshaping_stack.png)

Unstack, `unstack()`, como es de esperar, hace la misma operación, pero a la inversa.

![](https://pandas.pydata.org/docs/_images/reshaping_unstack.png)

Definamos un dataframe con multiíndice para demostrarlo:

In [ ]:
tuples = list(
    zip(
        *[
            ["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"],
            ["one", "two", "one", "two", "one", "two", "one", "two"],
        ]
    )
)
index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])
df = pd.DataFrame(np.random.randn(8, 2), index=index, columns=["A", "B"])
df

Hagamos stack para crear un dataset tidy:

In [ ]:
df = df.stack()
df

Esto ha creado automáticamente un índice de varios niveles, pero se puede volver a un índice numerado usando `df.reset_index()`

Ahora veamos unstack, pero en lugar de hacer unstack de las variables 'A' y 'B' con las que empezamos, hagamos unstack de la columna 'first' pasando `level=0` (por defecto se hace unstack del índice más interno). Este diagrama muestra lo que ocurre:

![](https://pandas.pydata.org/docs/_images/reshaping_unstack_0.png)

Y aquí está el código:

In [ ]:
df.unstack(level=0)

::: {.callout-tip title="Ejercicio"}
¿Qué ocurre si haces el unstack a `level=1`? ¿Y si aplicas `unstack()` dos veces?
:::

### Pivotar datos de largo a ancho

`pivot()` y `pivot_table()` te ayudan a ordenar datos en los que una única observación está dispersa en varias filas.

![](https://pandas.pydata.org/docs/_images/reshaping_pivot.png)


Aquí tienes un dataframe de ejemplo en el que las observaciones están repartidas en varias filas:

In [ ]:
df_tb_cp = pd.read_parquet(
    "https://github.com/aeturrell/python4DS/raw/refs/heads/main/data/who_tb_case_and_pop.parquet"
)
df_tb_cp.head()

Puedes ver que, para cada año-país, tenemos "case" y "population" en filas distintas.

Ahora hagamos pivot para ver la diferencia:

In [ ]:
pivoted = df_tb_cp.pivot(
    index=["country", "year"], columns=["type"], values="count"
).reset_index()
pivoted

Los pivots son especialmente útiles para datos de series temporales, en los que operaciones como `shift()` o `diff()` se aplican normalmente suponiendo que una entrada de una fila sigue (en el tiempo) a la de arriba. Cuando hacemos `shift()`, a menudo queremos desplazar una sola variable en el tiempo, pero si una única observación (en este caso, una fecha) está en varias filas, la temporalidad se va a desajustar. Veamos un ejemplo.

In [ ]:
import numpy as np

data = {
    "value": np.random.randn(20),
    "variable": ["A"] * 10 + ["B"] * 10,
    "date": (
        list(pd.date_range("1/1/2000", periods=10, freq="ME"))
        + list(pd.date_range("1/1/2000", periods=10, freq="ME"))
    ),
}
df = pd.DataFrame(data, columns=["date", "variable", "value"])
df.sample(5)

Si simplemente ejecutamos `shift()` sobre lo anterior, se desplazarán juntas las variables B y A, aunque se solapan en el tiempo y son variables distintas. Por eso pivotamos a un formato más ancho (y después ya podemos desplazar en el tiempo con seguridad).

In [ ]:
df.pivot(index="date", columns="variable", values="value").shift(1)

::: {.callout-tip title="Ejercicio"}
¿Por qué la primera entrada es NaN?
:::


::: {.callout-tip title="Ejercicio"}
Realiza un `pivot()` que se aplique tanto a la columna `variable` como a la columna `category` en el ejemplo anterior, donde category se define de modo que `df["category"] = np.random.choice(["type1", "type2", "type3", "type4"], 20). (Pista: recuerda que tendrás que pasar varios objetos mediante una lista).
:::